In [2]:
import chromadb
from chromadb.api.types import EmbeddingFunction
from typing import List, Sequence
from ollama import Client as OllamaClient
import pandas as pd

from embeders import OllamaEmbeddingFunction


In [3]:
ef = OllamaEmbeddingFunction()
client = chromadb.PersistentClient(path="./.chroma_db")

In [ ]:
# Use Ollama for both add and query embeddings
ef = OllamaEmbeddingFunction(model="mxbai-embed-large", host="http://127.0.0.1:11434")
collection = client.get_or_create_collection(name="checks", embedding_function=ef)

# Add docs (embeddings computed via Ollama)

df = pd.read_csv("./Export-2025-April-30.csv", sep=";")
df = df.dropna(subset=["Content"])

docs = df["Content"].tolist()
ids = [str(i) for i in df.index.tolist()]

batch_size = 128  # You can adjust this batch size as needed
num_batches = (len(docs) + batch_size - 1) // batch_size

for i in tqdm(range(num_batches), desc="Uploading batches"):
    batch_docs = docs[i*batch_size:(i+1)*batch_size]
    batch_ids = ids[i*batch_size:(i+1)*batch_size]
    collection.upsert(ids=batch_ids, documents=batch_docs)

In [ ]:
from tqdm import tqdm

client = chromadb.PersistentClient(
    path=r".\.chroma_db"
)

# Use Ollama for both add and query embeddings
ef = OllamaEmbeddingFunction(model="mxbai-embed-large", host="http://127.0.0.1:11434")
collection = client.get_or_create_collection(name="posts", embedding_function=ef)

# Add docs (embeddings computed via Ollama)
df_pubs = pd.read_excel(r".\2023_Completo_redem_0304.xlsx")
df_pubs = df_pubs.dropna(subset=["Message-ID", "Message"])

docs = df_pubs["Message"].tolist()
ids = [str(i) for i in df_pubs["Message-ID"].tolist()]

batch_size = 64  # You can adjust this batch size as needed
num_batches = (len(docs) + batch_size - 1) // batch_size

for i in tqdm(range(num_batches), desc="Uploading batches"):
    batch_docs = docs[i*batch_size:(i+1)*batch_size]
    batch_ids = ids[i*batch_size:(i+1)*batch_size]
    collection.upsert(ids=batch_ids, documents=batch_docs)

# # Query (query embedded via Ollama)
res = collection.query(query_texts=["2024"], n_results=2)
print(res)

Uploading batches:   4%|▍         | 234/5788 [1:53:23<38:57:42, 25.25s/it]